In [ ]:
import pandas as pd
import numpy as np

# Load the permeability data, specifying a space (' ') as the delimiter
csv_file_path = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/perm.csv'
permeability_data = pd.read_csv(csv_file_path, delim_whitespace=True, header=None)  # Use delim_whitespace for space-separated values

# Check how many columns are in the CSV
print(f"CSV contains {permeability_data.shape[1]} columns.")

# Assign new column names based on your description
expected_columns = ['i_index', 'j_index', 'k_index', 'x_coord', 'y_coord', 'z_coord', 'LithoProp']
if permeability_data.shape[1] == len(expected_columns):  # Check if the number of columns matches
    permeability_data.columns = expected_columns
else:
    # Handle the case where the column count does not match expectation
    raise ValueError(f"Unexpected number of columns in the CSV: {permeability_data.shape[1]}. Expected {len(expected_columns)}.")

# Force relevant columns to numeric, setting errors='coerce'
permeability_data['i_index'] = pd.to_numeric(permeability_data['i_index'], errors='coerce')
permeability_data['j_index'] = pd.to_numeric(permeability_data['j_index'], errors='coerce')
permeability_data['k_index'] = pd.to_numeric(permeability_data['k_index'], errors='coerce')
permeability_data['LithoProp'] = pd.to_numeric(permeability_data['LithoProp'], errors='coerce')

# Drop rows with any NaN values (non-numeric entries or missing data)
permeability_data = permeability_data.dropna()

# Convert indices to integers after cleaning
permeability_data['i_index'] = permeability_data['i_index'].astype(int)
permeability_data['j_index'] = permeability_data['j_index'].astype(int)
permeability_data['k_index'] = permeability_data['k_index'].astype(int)

# Replace -99 values in 'LithoProp' column with NaN for better handling of missing data
permeability_data['permeability'] = permeability_data['LithoProp'].replace(-99, np.nan)

# Get the max dimensions based on the indices
max_i = permeability_data['i_index'].max() + 1
max_j = permeability_data['j_index'].max() + 1
max_k = permeability_data['k_index'].max() + 1

# Create a 3D array with NaN for missing values
permeability_cube = np.full((max_i, max_j, max_k), np.nan)

# Populate the 3D array with permeability values
permeability_cube[permeability_data['i_index'], permeability_data['j_index'], permeability_data['k_index']] = permeability_data['permeability']

# Confirm dimensions and inspect the permeability_cube structure
print("Facies cube shape:", permeability_cube.shape)

# Creat colorful layers for whole model

# Facies

In [ ]:
# /home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/export_facies_layers_pal_only.py
from __future__ import annotations

import json
import math
import os
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont


# ----------------------------- Config -----------------------------

CSV_FILE_PATH = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Facies.csv'
OUTPUT_DIR = "/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_facies"

# Resolution controls
TARGET_LONG_SIDE_PX = 12000   # enforce pixel count
MIN_SCALE = 8                 # minimum integer upscale
PRINT_DPI = 600               # metadata only

BACKGROUND_INDEX = 255        # reserved for "no data" (black)

# Legend colors (matches your original)
LEGEND: List[Tuple[str, str]] = [
    ("Tight",            "#8B4513"),
    ("Very Poor Sand",   "#00FF00"),
    ("Poor Sand",        "#ADFF2F"),
    ("Fair sand",        "#FFFF00"),
    ("Good Sand",        "#FFA500"),
    ("Good Clean Sand",  "#FF00FF"),
    ("Very Good Sand",   "#FF0000"),
]

# If you know the exact raw-code → legend-index mapping, set it here (1-based legend index).
# Example: RAW_TO_LEGEND = {0:1, 1:2, 2:3, 3:4, 4:5, 5:6, 6:7}
RAW_TO_LEGEND: Dict[int, int] = {}


# --------------------------- Helpers ------------------------------

def hex_to_rgb(h: str) -> Tuple[int, int, int]:
    h = h.lstrip('#')
    return int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)

def build_palette() -> List[int]:
    """Why: fixed palette guarantees exact legend colors regardless of data order."""
    pal = [0] * (256 * 3)
    pal[BACKGROUND_INDEX * 3: BACKGROUND_INDEX * 3 + 3] = [0, 0, 0]
    for idx, (_, hx) in enumerate(LEGEND, start=1):
        r, g, b = hex_to_rgb(hx)
        s = idx * 3
        pal[s:s + 3] = [r, g, b]
    return pal

def dataframe_to_cube(df: pd.DataFrame) -> np.ndarray:
    mx_i, mx_j, mx_k = int(df['i'].max()) + 1, int(df['j'].max()) + 1, int(df['k'].max()) + 1
    cube = np.full((mx_i, mx_j, mx_k), np.nan, dtype=float)
    cube[df['i'], df['j'], df['k']] = df['LithoProp'].astype(float).values
    return cube

def make_mapping(unique_codes: List[int]) -> Dict[int, int]:
    """Why: avoid wrong colors when raw facies codes aren't 1..7."""
    if RAW_TO_LEGEND:
        missing = [c for c in unique_codes if c not in RAW_TO_LEGEND]
        if missing:
            print(f"WARNING: unmapped raw codes {missing} -> background")
        return RAW_TO_LEGEND.copy()
    mapping: Dict[int, int] = {}
    for idx, code in enumerate(sorted(unique_codes)[:len(LEGEND)], start=1):
        mapping[code] = idx
    return mapping

def compute_scale(w: int, h: int) -> int:
    need = TARGET_LONG_SIDE_PX / max(w, h) if TARGET_LONG_SIDE_PX else 1.0
    return max(MIN_SCALE, int(math.ceil(need)))

def export_layer(indexed: np.ndarray, k: int, palette: List[int]) -> None:
    imgP = Image.fromarray(indexed, mode="P")
    imgP.putpalette(palette)

    scale = compute_scale(imgP.width, imgP.height)
    if scale > 1:
        # NEAREST preserves classes; palette re-applied to keep exact colors.
        imgP = imgP.resize((imgP.width * scale, imgP.height * scale), resample=Image.NEAREST)
        imgP.putpalette(palette)

    out = os.path.join(OUTPUT_DIR, f"Facies_layer_{k}_pal_{imgP.width}x{imgP.height}.png")
    imgP.save(out, format="PNG", optimize=True, compress_level=9, dpi=(PRINT_DPI, PRINT_DPI))
    print(f"[PNG P]  {out} -> {imgP.width}x{imgP.height}px")

def legend_preview(palette: List[int]) -> str:
    w, h = 520, 30 * len(LEGEND) + 20
    im = Image.new("RGB", (w, h), (0, 0, 0))
    draw = ImageDraw.Draw(im)
    try:
        font = ImageFont.load_default()
    except Exception:
        font = None
    y = 10
    for i, (name, _) in enumerate(LEGEND, start=1):
        r, g, b = palette[i * 3: i * 3 + 3]
        draw.rectangle([10, y, 60, y + 20], fill=(r, g, b))
        draw.text((70, y + 3), f"{i} – {name}", (255, 255, 255), font)
        y += 30
    p = os.path.join(OUTPUT_DIR, "facies_legend_preview.png")
    im.save(p, format="PNG", optimize=True)
    return p


# ------------------------------ Main ------------------------------

def main() -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    df = pd.read_csv(CSV_FILE_PATH, skiprows=6, delim_whitespace=True, header=None)
    df = df[[0, 1, 2, 6]]
    df.columns = ['i', 'j', 'k', 'LithoProp']
    df = df.apply(pd.to_numeric, errors='coerce').dropna()
    df[['i', 'j', 'k', 'LithoProp']] = df[['i', 'j', 'k', 'LithoProp']].astype(int)

    counts = df['LithoProp'].value_counts().sort_index()
    mapping = make_mapping(counts.index.tolist())
    palette = build_palette()

    with open(os.path.join(OUTPUT_DIR, "facies_mapping_report.json"), "w") as f:
        json.dump({
            "unique_codes_sorted": counts.index.tolist(),
            "counts": counts.to_dict(),
            "mapping_raw_to_legend_index": mapping,
            "legend_index_to_label": {i + 1: LEGEND[i][0] for i in range(len(LEGEND))},
            "target_long_side_px": TARGET_LONG_SIDE_PX,
            "min_scale": MIN_SCALE,
            "print_dpi": PRINT_DPI,
            "background_index": BACKGROUND_INDEX,
        }, f, indent=2)

    print("Legend preview:", legend_preview(palette))

    cube = dataframe_to_cube(df)
    for k in range(cube.shape[2]):
        layer = cube[:, :, k]
        if np.isnan(layer).all():
            continue
        indexed = np.full(layer.shape, BACKGROUND_INDEX, dtype=np.uint8)
        with np.errstate(invalid='ignore'):
            for raw_code, legend_idx in mapping.items():
                m = (layer == raw_code)
                if np.any(m):
                    indexed[m] = np.uint8(legend_idx)
        export_layer(indexed, k, palette)

if __name__ == "__main__":
    main()


# Permeability

In [ ]:
# /home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/export_perm_layers.py
from __future__ import annotations

import json
import math
import os
from typing import List, Sequence, Tuple

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

# ------------------------- Configuration -------------------------

CSV_FILE_PATH = (
    "/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/perm.csv"
)
OUTPUT_DIR = (
    "/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_Perm"
)

# 0-based index of permeability column after we select columns from CSV
PERM_COL = 6

# Permeability range [mD]; used for log-scale mapping and legend
PERM_MIN = 0.01
PERM_MAX = 1000.0

# Resolution controls
TARGET_LONG_SIDE_PX = 12000  # long side in pixels
MIN_SCALE = 8                # minimum integer upscale
PRINT_DPI = 300
PNG_COMPRESS_LEVEL = 1       # lower = bigger file, better quality
BACKGROUND_INDEX = 255       # reserved for NODATA / outside grid

# RMS-style gradient (blue → cyan → green → yellow → orange → red)
GRADIENT_STOPS: Sequence[Tuple[float, str]] = (
    (0.00, "#0000FF"),  # deep blue (lowest)
    (0.25, "#00FFFF"),  # cyan
    (0.50, "#00FF00"),  # green
    (0.75, "#FFFF00"),  # yellow
    (0.90, "#FFA500"),  # orange
    (1.00, "#FF0000"),  # red (highest)
)

# 0..254 for ramp; 255 reserved for background
LUT_SIZE = 255

# ----------------------------- Utils -----------------------------


def hex_to_rgb(h: str) -> Tuple[int, int, int]:
    h = h.lstrip("#")
    return int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)


def build_lut(stops: Sequence[Tuple[float, str]], lut_size: int) -> np.ndarray:
    positions = np.array([p for p, _ in stops], dtype=float)
    colors = np.array([hex_to_rgb(c) for _, c in stops], dtype=float)
    x = np.linspace(0.0, 1.0, lut_size, dtype=float)

    r = np.interp(x, positions, colors[:, 0])
    g = np.interp(x, positions, colors[:, 1])
    b = np.interp(x, positions, colors[:, 2])

    return np.stack([r, g, b], axis=1).round().astype(np.uint8)


def dataframe_to_cube(df: pd.DataFrame) -> np.ndarray:
    mx_i = int(df["i"].max()) + 1
    mx_j = int(df["j"].max()) + 1
    mx_k = int(df["k"].max()) + 1

    cube = np.full((mx_i, mx_j, mx_k), np.nan, dtype=float)
    cube[df["i"], df["j"], df["k"]] = df["Permeability"].astype(float).values
    return cube


def compute_scale(w: int, h: int) -> int:
    needed = TARGET_LONG_SIDE_PX / max(w, h) if TARGET_LONG_SIDE_PX else 1.0
    return max(MIN_SCALE, int(math.ceil(needed)))


def put_palette(imgP: Image.Image, lut: np.ndarray) -> None:
    # keep index 255 as pure black (background)
    palette: List[int] = [0] * (256 * 3)
    flat = lut.reshape(-1).tolist()
    palette[0 : len(flat)] = flat
    palette[BACKGROUND_INDEX * 3 : BACKGROUND_INDEX * 3 + 3] = [0, 0, 0]
    imgP.putpalette(palette)


def legend_preview_png(lut: np.ndarray, out_dir: str, vmin: float, vmax: float) -> str:
    bar_w, bar_h = 80, 800
    idx = np.linspace(0, LUT_SIZE - 1, bar_h).astype(np.int16)
    grad = lut[idx][:, None, :]
    grad_img = np.repeat(grad, bar_w, axis=1).astype(np.uint8)
    legend = Image.fromarray(grad_img, mode="RGB")

    canvas_w, canvas_h = 360, legend.height + 60
    canvas = Image.new("RGB", (canvas_w, canvas_h), (0, 0, 0))
    canvas.paste(legend, (40, 40))

    draw = ImageDraw.Draw(canvas)
    try:
        font = ImageFont.load_default()
    except Exception:
        font = None

    # log-spaced ticks across [vmin, vmax]
    log_min = math.log10(vmin)
    log_max = math.log10(vmax)
    ticks = np.logspace(log_min, log_max, num=6)

    for t in ticks:
        y_rel = (math.log10(t) - log_min) / (log_max - log_min)
        y = 40 + int((1.0 - y_rel) * legend.height)
        draw.line(
            [(40 + bar_w + 6, y), (40 + bar_w + 20, y)],
            fill=(255, 255, 255),
        )
        draw.text(
            (40 + bar_w + 26, y - 6),
            f"{t:.2f}",
            fill=(255, 255, 255),
            font=font,
        )

    title = "Permeability [mD]"
    # Use textbbox instead of deprecated textsize
    bbox = draw.textbbox((0, 0), title, font=font)
    tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
    draw.text(((canvas_w - tw) // 2, 10), title, fill=(255, 255, 255), font=font)

    path = os.path.join(out_dir, "permeability_legend_preview.png")
    canvas.save(path, format="PNG", optimize=True)
    return path


# ------------------------------ Main ------------------------------


def main() -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Load CSV and select permeability
    df = pd.read_csv(CSV_FILE_PATH, skiprows=6, delim_whitespace=True, header=None)
    df = df[[0, 1, 2, PERM_COL]].copy()
    df.columns = ["i", "j", "k", "Permeability"]
    df = df.apply(pd.to_numeric, errors="coerce").dropna()
    df[["i", "j", "k"]] = df[["i", "j", "k"]].astype(int)

    cube = dataframe_to_cube(df)

    # Build LUT and legend once
    lut = build_lut(GRADIENT_STOPS, LUT_SIZE)
    print("Legend preview:", legend_preview_png(lut, OUTPUT_DIR, PERM_MIN, PERM_MAX))

    log_min = math.log10(PERM_MIN)
    log_max = math.log10(PERM_MAX)
    log_range = max(log_max - log_min, 1e-9)

    for k in range(cube.shape[2]):
        layer = cube[:, :, k]
        if np.isnan(layer).all():
            continue

        mask_valid = ~np.isnan(layer)

        # Clip values and map to log-space
        clipped = np.full_like(layer, PERM_MIN, dtype=float)
        clipped[mask_valid] = np.clip(layer[mask_valid], PERM_MIN, PERM_MAX)

        log_layer = np.zeros_like(clipped, dtype=float)
        log_layer[mask_valid] = np.log10(clipped[mask_valid])

        norm = (log_layer - log_min) / log_range
        indices = (np.clip(norm, 0.0, 1.0) * (LUT_SIZE - 1)).round().astype(np.uint8)

        indexed = np.where(mask_valid, indices, BACKGROUND_INDEX).astype(np.uint8)

        imgP = Image.fromarray(indexed, mode="P")
        put_palette(imgP, lut)

        scale = compute_scale(imgP.width, imgP.height)
        if scale > 1:
            imgP = imgP.resize(
                (imgP.width * scale, imgP.height * scale),
                resample=Image.NEAREST,
            )
            put_palette(imgP, lut)

        outP = os.path.join(
            OUTPUT_DIR,
            f"Permeability_layer_{k}_pal_{imgP.width}x{imgP.height}.png",
        )
        imgP.save(
            outP,
            format="PNG",
            optimize=True,
            compress_level=PNG_COMPRESS_LEVEL,
            dpi=(PRINT_DPI, PRINT_DPI),
        )
        print(f"[PNG P] {outP}")

    report = {
        "csv": CSV_FILE_PATH,
        "perm_col_index": PERM_COL,
        "range": [PERM_MIN, PERM_MAX],
        "log_scale": True,
        "lut_size": LUT_SIZE,
        "target_long_side_px": TARGET_LONG_SIDE_PX,
        "min_scale": MIN_SCALE,
        "print_dpi": PRINT_DPI,
        "background_index": BACKGROUND_INDEX,
        "gradient_stops": list(GRADIENT_STOPS),
    }
    with open(os.path.join(OUTPUT_DIR, "permeability_export_report.json"), "w") as f:
        json.dump(report, f, indent=2)


if __name__ == "__main__":
    main()


# Porosity

In [ ]:
# /home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/export_porosity_layers.py
"""
Export porosity layers as high-resolution images with an exact legend-matched rainbow gradient.
- Continuous property (porosity only).
- Palette-indexed PNG for exact colors + optional RGB PNG for larger file size.
- NEAREST resampling preserves cell colors on upscale.
"""

from __future__ import annotations

import json
import math
import os
from typing import Iterable, List, Sequence, Tuple

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont


# ------------------------- Configuration -------------------------

CSV_FILE_PATH = "/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Porosity.csv"  # <-- change if needed
OUTPUT_DIR = "/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_Porosity"

# Which column in the CSV holds porosity (0-based index after selecting columns)?
# If your file layout differs, set PORO_COL to the correct index or adapt the column selection below.
PORO_COL = 6  # assumes CSV columns: i j k ... PORO at col 6 (like your Facies.csv)

# Value range for the legend; clamp outside into [MIN, MAX]
PORO_MIN = 0.0
PORO_MAX = 0.225  # matches your legend top

# Resolution controls (choose either/both)
TARGET_LONG_SIDE_PX = 12000   # guarantees >= this many pixels on the long side
MIN_SCALE = 8                 # minimal integer upscale even if target already reached
PRINT_DPI = 600               # metadata only; does not change pixel count
PNG_COMPRESS_LEVEL = 9
BACKGROUND_INDEX = 255       # reserved for NODATA (black)

# Legend gradient (bottom→top)
GRADIENT_STOPS: Sequence[Tuple[float, str]] = (
    (0.00, "#FF00FF"),  # magenta
    (0.10, "#0000FF"),  # blue
    (0.20, "#00FFFF"),  # cyan
    (0.40, "#00FF00"),  # green
    (0.70, "#FFFF00"),  # yellow
    (0.85, "#FFA500"),  # orange
    (1.00, "#FF0000"),  # red
)

LUT_SIZE = 255  # 0..254 for ramp; 255 reserved for background


# ----------------------------- Utils -----------------------------

def hex_to_rgb(h: str) -> Tuple[int, int, int]:
    h = h.lstrip("#")
    return int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)

def build_lut(stops: Sequence[Tuple[float, str]], lut_size: int) -> np.ndarray:
    """Why: 255-color ramp leaves index 255 for background."""
    pos = np.array([p for p, _ in stops], dtype=float)
    rgb = np.array([hex_to_rgb(c) for _, c in stops], dtype=float)
    x = np.linspace(0.0, 1.0, lut_size, dtype=float)
    r = np.interp(x, pos, rgb[:, 0])
    g = np.interp(x, pos, rgb[:, 1])
    b = np.interp(x, pos, rgb[:, 2])
    return np.stack([r, g, b], axis=1).round().astype(np.uint8)  # (lut_size, 3)

def dataframe_to_cube(df: pd.DataFrame) -> np.ndarray:
    mx_i = int(df["i"].max()) + 1
    mx_j = int(df["j"].max()) + 1
    mx_k = int(df["k"].max()) + 1
    cube = np.full((mx_i, mx_j, mx_k), np.nan, dtype=float)
    cube[df["i"], df["j"], df["k"]] = df["Porosity"].astype(float).values
    return cube

def compute_scale(w: int, h: int) -> int:
    need = TARGET_LONG_SIDE_PX / max(w, h) if TARGET_LONG_SIDE_PX else 1.0
    return max(MIN_SCALE, int(math.ceil(need)))

def put_palette(imgP: Image.Image, lut: np.ndarray) -> None:
    """Why: Ensure background index 255 stays black."""
    pal = [0] * (256 * 3)
    flat = lut.reshape(-1).tolist()           # fills indices 0..254
    pal[0:len(flat)] = flat
    pal[BACKGROUND_INDEX * 3: BACKGROUND_INDEX * 3 + 3] = [0, 0, 0]
    imgP.putpalette(pal)

def legend_preview_png(lut: np.ndarray, out_dir: str, vmin: float, vmax: float) -> str:
    bar_w, bar_h = 60, 600
    idx = np.linspace(0, LUT_SIZE - 1, bar_h).astype(np.int16)
    grad = lut[idx][:, None, :]
    grad_img = np.repeat(grad, bar_w, axis=1).astype(np.uint8)
    legend = Image.fromarray(grad_img, mode="RGB").transpose(Image.ROTATE_270).transpose(Image.FLIP_LEFT_RIGHT)
    canvas_w, canvas_h = 350, legend.height + 40
    canvas = Image.new("RGB", (canvas_w, canvas_h), (0, 0, 0))
    canvas.paste(legend, (20, 20))
    draw = ImageDraw.Draw(canvas)
    try:
        font = ImageFont.load_default()
    except Exception:
        font = None
    ticks = np.linspace(vmin, vmax, 10)
    for t in ticks:
        y_rel = (t - vmin) / max(vmax - vmin, 1e-9)
        y = 20 + int((1.0 - y_rel) * legend.height)
        draw.line([(20 + legend.width + 6, y), (20 + legend.width + 20, y)], fill=(255, 255, 255))
        draw.text((20 + legend.width + 26, y - 6), f"{t:.3f}", fill=(255, 255, 255), font=font)
    path = os.path.join(out_dir, "porosity_legend_preview.png")
    canvas.save(path, format="PNG", optimize=True)
    return path


# ------------------------------ Main ------------------------------

def main() -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Load CSV and select porosity
    df = pd.read_csv(CSV_FILE_PATH, skiprows=6, delim_whitespace=True, header=None)
    df = df[[0, 1, 2, PORO_COL]].copy()
    df.columns = ["i", "j", "k", "Porosity"]
    df = df.apply(pd.to_numeric, errors="coerce").dropna()
    df[["i", "j", "k"]] = df[["i", "j", "k"]].astype(int)

    cube = dataframe_to_cube(df)

    # Build LUT and preview
    lut = build_lut(GRADIENT_STOPS, LUT_SIZE)
    print("Legend preview:", legend_preview_png(lut, OUTPUT_DIR, PORO_MIN, PORO_MAX))

    # Export each layer as palette PNG only
    for k in range(cube.shape[2]):
        layer = cube[:, :, k]
        if np.isnan(layer).all():
            continue

        norm = (layer - PORO_MIN) / max(PORO_MAX - PORO_MIN, 1e-9)
        idx = (np.clip(norm, 0.0, 1.0) * (LUT_SIZE - 1)).round().astype(np.uint8)  # 0..254
        indexed = np.where(np.isnan(layer), BACKGROUND_INDEX, idx).astype(np.uint8)

        imgP = Image.fromarray(indexed, mode="P")
        put_palette(imgP, lut)

        scale = compute_scale(imgP.width, imgP.height)
        if scale > 1:
            imgP = imgP.resize((imgP.width * scale, imgP.height * scale), resample=Image.NEAREST)
            put_palette(imgP, lut)  # re-apply to avoid any palette loss

        outP = os.path.join(OUTPUT_DIR, f"Porosity_layer_{k}_pal_{imgP.width}x{imgP.height}.png")
        imgP.save(outP, format="PNG", optimize=True, compress_level=PNG_COMPRESS_LEVEL, dpi=(PRINT_DPI, PRINT_DPI))
        print(f"[PNG P] {outP}")

    # Run report
    report = {
        "csv": CSV_FILE_PATH,
        "poro_col_index": PORO_COL,
        "range": [PORO_MIN, PORO_MAX],
        "lut_size": LUT_SIZE,
        "target_long_side_px": TARGET_LONG_SIDE_PX,
        "min_scale": MIN_SCALE,
        "print_dpi": PRINT_DPI,
        "background_index": BACKGROUND_INDEX,
        "gradient_stops": list(GRADIENT_STOPS),
    }
    with open(os.path.join(OUTPUT_DIR, "porosity_export_report.json"), "w") as f:
        json.dump(report, f, indent=2)

if __name__ == "__main__":
    main()


# Vshale

In [ ]:
# /home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/export_vshale_layers_pal_only.py
"""
VShale -> palette-only PNGs with legend-matched ramp (red→orange→yellow→green→cyan→blue→magenta).
- Value range: 0.00 .. 0.50 (clamped).
- 255-color LUT; palette index 255 = black background.
- NEAREST upscale; no RGB/TIFF copies.
"""

from __future__ import annotations

import json
import math
import os
from typing import Sequence, Tuple

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont


# ------------------------- Configuration -------------------------

CSV_FILE_PATH = "/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Vshale_ns.csv"
OUTPUT_DIR = "/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_VShale"

VSH_COL = 6              # column index of VShale in your CSV (adjust if needed)
VSH_MIN = 0.00
VSH_MAX = 0.50

TARGET_LONG_SIDE_PX = 12000
MIN_SCALE = 8
PRINT_DPI = 600
PNG_COMPRESS_LEVEL = 9
BACKGROUND_INDEX = 255

# Gradient stops normalized to [0,1] of the range [VSH_MIN, VSH_MAX].
# Matches your legend: bottom red (0.00) → top magenta (0.50).
GRADIENT_STOPS: Sequence[Tuple[float, str]] = (
    (0.00, "#FF0000"),  # red      @ 0.00
    (0.10, "#FFA500"),  # orange   @ 0.05
    (0.30, "#FFFF00"),  # yellow   @ 0.15
    (0.50, "#00FF00"),  # green    @ 0.25
    (0.60, "#00FFFF"),  # cyan     @ 0.30
    (0.90, "#0000FF"),  # blue     @ 0.45
    (1.00, "#FF00FF"),  # magenta  @ 0.50
)

LUT_SIZE = 255  # 0..254 ramp; 255 background


# ----------------------------- Utils -----------------------------

def hex_to_rgb(h: str) -> tuple[int, int, int]:
    h = h.lstrip("#")
    return int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)

def build_lut(stops: Sequence[Tuple[float, str]], lut_size: int) -> np.ndarray:
    pos = np.array([p for p, _ in stops], dtype=float)
    rgb = np.array([hex_to_rgb(c) for _, c in stops], dtype=float)
    x = np.linspace(0.0, 1.0, lut_size, dtype=float)
    r = np.interp(x, pos, rgb[:, 0])
    g = np.interp(x, pos, rgb[:, 1])
    b = np.interp(x, pos, rgb[:, 2])
    return np.stack([r, g, b], axis=1).round().astype(np.uint8)

def dataframe_to_cube(df: pd.DataFrame) -> np.ndarray:
    mx_i, mx_j, mx_k = int(df["i"].max()) + 1, int(df["j"].max()) + 1, int(df["k"].max()) + 1
    cube = np.full((mx_i, mx_j, mx_k), np.nan, dtype=float)
    cube[df["i"], df["j"], df["k"]] = df["VShale"].astype(float).values
    return cube

def compute_scale(w: int, h: int) -> int:
    need = TARGET_LONG_SIDE_PX / max(w, h) if TARGET_LONG_SIDE_PX else 1.0
    return max(MIN_SCALE, int(math.ceil(need)))

def put_palette(imgP: Image.Image, lut: np.ndarray) -> None:
    pal = [0] * (256 * 3)
    flat = lut.reshape(-1).tolist()  # indices 0..254
    pal[0:len(flat)] = flat
    pal[BACKGROUND_INDEX * 3: BACKGROUND_INDEX * 3 + 3] = [0, 0, 0]
    imgP.putpalette(pal)

def legend_preview_png(lut: np.ndarray, out_dir: str, vmin: float, vmax: float) -> str:
    bar_w, bar_h = 60, 600
    idx = np.linspace(0, LUT_SIZE - 1, bar_h).astype(np.int16)
    grad = lut[idx][:, None, :]
    grad_img = np.repeat(grad, bar_w, axis=1).astype(np.uint8)
    legend = Image.fromarray(grad_img, mode="RGB").transpose(Image.ROTATE_270).transpose(Image.FLIP_LEFT_RIGHT)

    canvas_w, canvas_h = 350, legend.height + 40
    canvas = Image.new("RGB", (canvas_w, canvas_h), (0, 0, 0))
    canvas.paste(legend, (20, 20))
    draw = ImageDraw.Draw(canvas)
    try:
        font = ImageFont.load_default()
    except Exception:
        font = None

    ticks = np.linspace(vmin, vmax, 11)  # 0.00..0.50 step 0.05
    for t in ticks:
        y_rel = (t - vmin) / max(vmax - vmin, 1e-9)
        y = 20 + int((1.0 - y_rel) * legend.height)
        draw.line([(20 + legend.width + 6, y), (20 + legend.width + 20, y)], fill=(255, 255, 255))
        draw.text((20 + legend.width + 26, y - 6), f"{t:.2f}", fill=(255, 255, 255), font=font)

    path = os.path.join(out_dir, "vshale_legend_preview.png")
    canvas.save(path, format="PNG", optimize=True)
    return path


# ------------------------------ Main ------------------------------

def main() -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    df = pd.read_csv(CSV_FILE_PATH, skiprows=6, delim_whitespace=True, header=None)
    df = df[[0, 1, 2, VSH_COL]].copy()
    df.columns = ["i", "j", "k", "VShale"]
    df = df.apply(pd.to_numeric, errors="coerce").dropna()
    df[["i", "j", "k"]] = df[["i", "j", "k"]].astype(int)

    cube = dataframe_to_cube(df)

    lut = build_lut(GRADIENT_STOPS, LUT_SIZE)
    print("Legend preview:", legend_preview_png(lut, OUTPUT_DIR, VSH_MIN, VSH_MAX))

    for k in range(cube.shape[2]):
        layer = cube[:, :, k]
        if np.isnan(layer).all():
            continue

        norm = (layer - VSH_MIN) / max(VSH_MAX - VSH_MIN, 1e-9)
        idx = (np.clip(norm, 0.0, 1.0) * (LUT_SIZE - 1)).round().astype(np.uint8)  # 0..254
        indexed = np.where(np.isnan(layer), BACKGROUND_INDEX, idx).astype(np.uint8)

        imgP = Image.fromarray(indexed, mode="P")
        put_palette(imgP, lut)

        scale = compute_scale(imgP.width, imgP.height)
        if scale > 1:
            imgP = imgP.resize((imgP.width * scale, imgP.height * scale), resample=Image.NEAREST)
            put_palette(imgP, lut)

        outP = os.path.join(OUTPUT_DIR, f"VShale_layer_{k}_pal_{imgP.width}x{imgP.height}.png")
        imgP.save(outP, format="PNG", optimize=True, compress_level=PNG_COMPRESS_LEVEL, dpi=(PRINT_DPI, PRINT_DPI))
        print(f"[PNG P] {outP}")

    report = {
        "csv": CSV_FILE_PATH,
        "vsh_col_index": VSH_COL,
        "range": [VSH_MIN, VSH_MAX],
        "lut_size": LUT_SIZE,
        "target_long_side_px": TARGET_LONG_SIDE_PX,
        "min_scale": MIN_SCALE,
        "print_dpi": PRINT_DPI,
        "background_index": BACKGROUND_INDEX,
        "gradient_stops": list(GRADIENT_STOPS),
    }
    with open(os.path.join(OUTPUT_DIR, "vshale_export_report.json"), "w") as f:
        json.dump(report, f, indent=2)

if __name__ == "__main__":
    main()


# Sw